# ORO walkthrough

Infers the object-role ontology (ORO) of the paper's four OCEL 2.0 logs (bottom-up), checks the
Section 7 schemata against them (top-down), and exports each ORO as an OWL 2 ontology (`.ttl`).

Runs in the dev container (`.devcontainer/`), where Ontop serves the virtual knowledge graph at
`http://ontop:8080`. Ontop serves one log at a time: `load()` and `.serve()` stage a log and wait
until Ontop serves it.

In [1]:
import pandas as pd

from ocel_inference import SCHEMATA, load
from ocel_inference.datasets import path
from ocel_inference.diagnostics import transformation_coverage

## The axiom schemata

Each schema of the paper is one object (`ocel_inference.schemata`): its free variables, a
description, its DL axiom and, for bottom-up schemata, the query that finds every satisfied
instantiation. The counterexample query of an instantiation is generated from its DL axiom
(Table X: counterexamples to C ⊑ D are the answers to τ(C ⊓ ¬D)).

In [2]:
pd.DataFrame(
    [(s.label, ", ".join(s.free_variables), s.dl_axiom, s.mode, s.description) for s in SCHEMATA],
    columns=["schema", "free variables", "DL axiom", "primary use", "description"],
).style.hide(axis="index").set_properties(**{"text-align": "left"})

schema,free variables,DL axiom,primary use,description
init,"ot, a",R_a(ot) ⊑ R_start,bottom-up,Activity a consistently starts the lifecycle of ot-objects: every role of ot in a is a starting role.
clos,"ot, a",R_a(ot) ⊑ R_end,bottom-up,Activity a consistently ends the lifecycle of ot-objects: every role of ot in a is an ending role.
ext,ot,O_ot ⊑ ∀playsRole.¬(R_init(ot) ⊔ R_close(ot)),bottom-up,ot-objects play no init or closure role: their lifecycle starts and ends outside the log.
eres,ot,O_ot ⊓ (≥2 playsRole.R_a(ot)) ⋢ ⊥ for every a ∈ A_ot,bottom-up,"ot is an external resource type: all its role concepts are re-usable, i.e. no activity a has a role of ot that is single-use across the log (reuse(ot, a) for every a ∈ A_ot)."
trans_life,"ot_in, ot_out, a",R_a(ot_in) ⊑ R_end; R_a(ot_out) ⊑ R_start,bottom-up,"Activity a closes ot_in-objects and initializes ot_out-objects (clos(ot_in, a) and init(ot_out, a))."
oto,"ot_in, ot_out, a",T_a ⊑ (=1 frames.R_a(ot_in)); T_a ⊑ (=1 frames.R_a(ot_out)),top-down,Each transforming a-event frames exactly one ot_in-object and one ot_out-object.
split,"ot_in, ot_out, a",T_a ⊑ (=1 frames.R_a(ot_in)); T_a ⊑ (≥2 frames.R_a(ot_out)),top-down,Each transforming a-event frames one ot_in-object and at least two ot_out-objects.
merge,"ot_in, ot_out, a",T_a ⊑ (≥2 frames.R_a(ot_in)); T_a ⊑ (=1 frames.R_a(ot_out)),top-down,Each transforming a-event frames at least two ot_in-objects and one ot_out-object.
gtrans,"ot_in, ot_out, a",T_a ⊑ (≥1 frames.R_a(ot_in)); T_a ⊑ (≥1 frames.R_a(ot_out)),top-down,Each transforming a-event frames at least one ot_in-object and one ot_out-object.
trans_nec,"ot_in, ot_out, a",O_ot_in ⊑ (=1 playsRole.R_a(ot_in)); O_ot_out ⊑ (=1 playsRole.R_a(ot_out)),top-down,"Every ot_in-object is transformed in a, and every ot_out-object results from a, exactly once."


## 1. Bottom-up: inferred axioms

`load(name)` reads the OCEL 2.0 JSON, builds the object interaction graph (OIG) and the ORO's role
individuals in DuckDB, generates the Ontop mapping and waits for Ontop. With `visualize=True` it
also writes the OIG and ORO graphs (`oig_graph.html`, `oro_graph.html`) to `output/<log>/`.
`infer_and_display_role_categories()` lists every satisfied instantiation with its DL axiom; with
`export_oro=True` it also writes the ORO as OWL 2 to `output/<log>/oro.ttl`.

In [3]:
logistics = load("logistics", visualize=True)
axioms = logistics.infer_and_display_role_categories(export_oro=True)

ContainerLogistics.json: 35372 events, inference in 3.46 s


Ontop is serving ContainerLogistics.json.


34 inferred axioms


object type,activity,schema,axiom
container,Order Empty Containers,init,R_{Order Empty Containers}(container) ⊑ R_start
container,Depart,clos,R_Depart(container) ⊑ R_end
container,Bring to Loading Bay,single,O_container ⊑ (≤1 playsRole.R_{Bring to Loading Bay}(container))
container,Drive to Terminal,single,O_container ⊑ (≤1 playsRole.R_{Drive to Terminal}(container))
container,Load to Vehicle,single,O_container ⊑ (≤1 playsRole.R_{Load to Vehicle}(container))
container,Pick Up Empty Container,single,O_container ⊑ (≤1 playsRole.R_{Pick Up Empty Container}(container))
container,Place in Stock,single,O_container ⊑ (≤1 playsRole.R_{Place in Stock}(container))
container,Reschedule Container,single,O_container ⊑ (≤1 playsRole.R_{Reschedule Container}(container))
container,Weigh,single,O_container ⊑ (≤1 playsRole.R_Weigh(container))
container,Load Truck,reuse,O_container ⊓ (≥2 playsRole.R_{Load Truck}(container)) ⋢ ⊥


ORO written to output/logistics/oro.ttl (100 schema instantiations)


In [4]:
p2p = load("p2p", visualize=True)
axioms = p2p.infer_and_display_role_categories(export_oro=True)

ocel2-p2p.json: 14671 events, inference in 2.56 s


Ontop is serving ocel2-p2p.json.


40 inferred axioms


object type,activity,schema,axiom
goods_receipt,,ext,O_{goods_receipt} ⊑ ∀playsRole.¬(R_init(goods_receipt) ⊔ R_close(goods_receipt))
goods_receipt,Create Invoice Receipt,single,O_{goods_receipt} ⊑ (≤1 playsRole.R_{Create Invoice Receipt}(goods_receipt))
goods_receipt,Perform Two-Way Match,single,O_{goods_receipt} ⊑ (≤1 playsRole.R_{Perform Two-Way Match}(goods_receipt))
goods_receipt,Create Goods Receipt,reuse,O_{goods_receipt} ⊓ (≥2 playsRole.R_{Create Goods Receipt}(goods_receipt)) ⋢ ⊥
goods_receipt,Execute Payment,reuse,O_{goods_receipt} ⊓ (≥2 playsRole.R_{Execute Payment}(goods_receipt)) ⋢ ⊥
invoice_receipt,,ext,O_{invoice_receipt} ⊑ ∀playsRole.¬(R_init(invoice_receipt) ⊔ R_close(invoice_receipt))
invoice_receipt,,eres,O_{invoice_receipt} ⊓ (≥2 playsRole.R_a(invoice_receipt)) ⋢ ⊥ for every a ∈ A_{invoice_receipt}
invoice_receipt,Create Invoice Receipt,reuse,O_{invoice_receipt} ⊓ (≥2 playsRole.R_{Create Invoice Receipt}(invoice_receipt)) ⋢ ⊥
invoice_receipt,Execute Payment,reuse,O_{invoice_receipt} ⊓ (≥2 playsRole.R_{Execute Payment}(invoice_receipt)) ⋢ ⊥
invoice_receipt,Perform Two-Way Match,reuse,O_{invoice_receipt} ⊓ (≥2 playsRole.R_{Perform Two-Way Match}(invoice_receipt)) ⋢ ⊥


ORO written to output/p2p/oro.ttl (111 schema instantiations)


In [5]:
order_mgmt = load("order_mgmt", visualize=True)
axioms = order_mgmt.infer_and_display_role_categories(export_oro=True)

order-management.json: 21008 events, inference in 7.86 s


Ontop is serving order-management.json.


46 inferred axioms


object type,activity,schema,axiom
customers,,ext,O_customers ⊑ ∀playsRole.¬(R_init(customers) ⊔ R_close(customers))
customers,,eres,O_customers ⊓ (≥2 playsRole.R_a(customers)) ⋢ ⊥ for every a ∈ A_customers
customers,confirm order,reuse,O_customers ⊓ (≥2 playsRole.R_{confirm order}(customers)) ⋢ ⊥
customers,place order,reuse,O_customers ⊓ (≥2 playsRole.R_{place order}(customers)) ⋢ ⊥
employees,,ext,O_employees ⊑ ∀playsRole.¬(R_init(employees) ⊔ R_close(employees))
employees,,eres,O_employees ⊓ (≥2 playsRole.R_a(employees)) ⋢ ⊥ for every a ∈ A_employees
employees,confirm order,reuse,O_employees ⊓ (≥2 playsRole.R_{confirm order}(employees)) ⋢ ⊥
employees,create package,reuse,O_employees ⊓ (≥2 playsRole.R_{create package}(employees)) ⋢ ⊥
employees,failed delivery,reuse,O_employees ⊓ (≥2 playsRole.R_{failed delivery}(employees)) ⋢ ⊥
employees,item out of stock,reuse,O_employees ⊓ (≥2 playsRole.R_{item out of stock}(employees)) ⋢ ⊥


ORO written to output/order_mgmt/oro.ttl (131 schema instantiations)


In [6]:
bpi17 = load("bpi17", visualize=True)
axioms = bpi17.infer_and_display_role_categories(export_oro=True)

bpi17_ocel.json: 23561 events, inference in 2.81 s


Ontop is serving bpi17_ocel.json.


65 inferred axioms


object type,activity,schema,axiom
application,Create Application_complete,init,R_{Create Application_complete}(application) ⊑ R_start
application,Assess potential fraud_complete,single,O_application ⊑ (≤1 playsRole.R_{Assess potential fraud_complete}(application))
application,Call after offers_complete,single,O_application ⊑ (≤1 playsRole.R_{Call after offers_complete}(application))
application,Complete application_complete,single,O_application ⊑ (≤1 playsRole.R_{Complete application_complete}(application))
application,Complete_complete,single,O_application ⊑ (≤1 playsRole.R_{Complete_complete}(application))
application,Concept_complete,single,O_application ⊑ (≤1 playsRole.R_{Concept_complete}(application))
application,Denied_complete,single,O_application ⊑ (≤1 playsRole.R_{Denied_complete}(application))
application,Handle leads_complete,single,O_application ⊑ (≤1 playsRole.R_{Handle leads_complete}(application))
application,Pending_complete,single,O_application ⊑ (≤1 playsRole.R_{Pending_complete}(application))
application,Submitted_complete,single,O_application ⊑ (≤1 playsRole.R_{Submitted_complete}(application))


ORO written to output/bpi17/oro.ttl (190 schema instantiations)


## 2. Top-down: checking schemata (Section 7)

`check(schema, object_type, ...)` returns the counterexamples of an instantiated schema; an empty
table means it holds. Labels are the log's own activity and object-type names. `axiom(...)` shows the
DL axiom and `counterexample_query(...)` the SPARQL generated from it (same arguments).

| schema | free variables (keyword) |
| --- | --- |
| `init`, `clos`, `single` | `ot`, `a` (`activity`) |
| `alt` | `ot`, `A` (`activities`) |
| `budg` | `ot`, `A` (`activities`), `N` (`budget`) |
| `excl`, `shar`, `carr`, `carr_next`, `carr_fwd` | `ot`, `a1` (`sources`), `a2` (`targets`) |
| `ext`, `eres` | `ot` |
| `trans_life`, `oto`, `split`, `merge`, `gtrans`, `trans_nec` | `ot_in`, `ot_out`, `a` |

### Container Logistics

In [7]:
logistics.serve()

# Q1 init(Forklift, Weigh): the Weigh roles of forklifts that are not starting roles.
print(logistics.axiom("init", otype="Forklift", activity="Weigh"))
q1 = logistics.check("init", "Forklift", activity="Weigh")
q1.groupby("obj").size().rename("late Weigh events")

Ontop is serving ContainerLogistics.json.
R_Weigh(forklift) ⊑ R_start


obj
fl1    683
fl2    654
fl3    649
Name: late Weigh events, dtype: int64

In [8]:
# Q4 budg(Transport Document, {Reschedule Container, Depart}, 2)
print(logistics.axiom("budg", otype="Transport Document", activities=["Reschedule Container", "Depart"], budget=2))
logistics.check("budg", "Transport Document", activities=["Reschedule Container", "Depart"], budget=2)

O_{transport_document} ⊑ (≤2 playsRole.(R_{Reschedule Container}(transport_document) ⊔ R_Depart(transport_document)))


,obj
0,td101
1,td103
2,td104
3,td107
4,td111
5,td113
6,td115
7,td116
8,td123
9,td124


The SPARQL of Q1, generated from its axiom: `R_Weigh(forklift) ⊓ ¬R_start`, with R_start expanded to `Role ⊓ ¬∃nextRole⁻.Role`.

In [9]:
print(logistics.counterexample_query("init", otype="Forklift", activity="Weigh"))

PREFIX ex: <http://www.example.com/>
PREFIX oro: <http://www.example.com/oro/>
PREFIX ocel: <https://stl.mie.utoronto.ca/ontologies/ocel/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?obj ?event ?activity ?timestamp
WHERE {
  { SELECT DISTINCT ?x WHERE {
    ?x a <http://www.example.com/oro/R/forklift/Weigh> .
    MINUS {
        ?x a <http://www.example.com/oro/Role> .
        MINUS {
            ?y0 <http://www.example.com/oro/nextRole> ?x .
            ?y0 a <http://www.example.com/oro/Role> .
        }
    }
  } }
  ?obj oro:playsRole ?x .
  ?event oro:frames ?x ;
         ocel:hasActivity ?activity ;
         ocel:timestamp ?timestamp .
}
ORDER BY ?obj ?timestamp ?event


Is *Create Transport Document* a necessary transformation of Customer Order into Transport Document?
This reads the raw JSON: objects without events never reach the VKG.

In [10]:
coverage = transformation_coverage(
    path("logistics"), "Customer Order", "Transport Document",
    activity="Create Transport Document", o2o_qualifier="TD for CO",
)
coverage.report()
display(coverage.untransformed_sources, coverage.uncreated_targets)

Log end: 2024-08-21 16:20:39+00:00
(a) 594/600 Customer Order objects are transformed into a Transport Document
(b) 594/596 Transport Document objects are created from a Customer Order
Transformation pairs from events: 594, from O2O: 594, in only one of the two: 0


,events,first_activity,last_activity,last_timestamp,days_before_log_end
obj,,,,,
co587,1,Register Customer Order,Register Customer Order,2024-08-08 16:51:34+00:00,12.978530
co591,1,Register Customer Order,Register Customer Order,2024-08-13 09:30:54+00:00,8.284549
co595,1,Register Customer Order,Register Customer Order,2024-08-15 09:54:54+00:00,6.267882
co597,1,Register Customer Order,Register Customer Order,2024-08-16 11:57:38+00:00,5.182650
co598,1,Register Customer Order,Register Customer Order,2024-08-19 08:38:40+00:00,2.320822
co599,1,Register Customer Order,Register Customer Order,2024-08-19 14:42:02+00:00,2.068484


,events,first_activity,last_activity,last_timestamp,days_before_log_end,o2o_linked_source,latest_attribute_time,attribute_after_log_end
obj,,,,,,,,
td595,0,NaN,NaN,NaT,NaN,False,2024-08-22 09:42:23+00:00,True
td596,0,NaN,NaN,NaT,NaN,False,2024-08-22 13:41:43+00:00,True


### BPI 2017

In [11]:
bpi17.serve()

# Q2 alt(Offer, {Accepted, Cancelled, Refused}): offers with none or several of the three closures.
CLOSURES = ["Accepted_complete", "Cancelled_complete", "Refused_complete"]
print(bpi17.axiom("alt", otype="offer", activities=CLOSURES))
q2 = bpi17.check("alt", "offer", activities=CLOSURES)
q2

Ontop is serving bpi17_ocel.json.
O_offer ⊑ (=1 playsRole.(R_{Accepted_complete}(offer) ⊔ R_{Cancelled_complete}(offer) ⊔ R_{Refused_complete}(offer)))


,obj
0,Offer_1033292322
1,Offer_1104554024
2,Offer_1318797754
3,Offer_1770894681


In [12]:
# What happened to these offers? Their events and those of their application and workflow.
for offer in q2["obj"]:
    timeline = bpi17.object_timeline(offer, related_types=["application", "workflow"])
    print(f"{offer}: {len(timeline)} events, last: {timeline['activity'].iloc[-1]}")
    display(timeline.tail(8))

Offer_1033292322: 33 events, last: Sent (online only)_complete


,timestamp,event,activity,object
25,2017-02-01T10:07:53+00:00,E_1161423,Validating_complete,Application_1170192930
26,2017-02-01T10:07:56+00:00,E_1161424,Validate application_complete,Application_1170192930
27,2017-02-01T10:07:56+00:00,E_1161427,Incomplete_complete,Application_1170192930
28,2017-02-01T11:23:02+00:00,E_1161429,Create Offer_complete,Application_1170192930
29,2017-02-01T11:23:03+00:00,E_1161430,Created_complete,Application_1170192930
30,2017-02-01T11:23:03+00:00,E_1161430,Created_complete,Offer_1033292322
31,2017-02-01T11:23:42+00:00,E_1161431,Sent (online only)_complete,Application_1170192930
32,2017-02-01T11:23:42+00:00,E_1161431,Sent (online only)_complete,Offer_1033292322


Offer_1104554024: 12 events, last: Complete_complete


,timestamp,event,activity,object
4,2017-01-02T12:10:41+00:00,E_1200350,Accepted_complete,Application_1516032553
5,2017-01-02T12:14:22+00:00,E_1200351,Create Offer_complete,Application_1516032553
6,2017-01-02T12:14:22+00:00,E_1200352,Created_complete,Application_1516032553
7,2017-01-02T12:14:22+00:00,E_1200352,Created_complete,Offer_1104554024
8,2017-01-02T12:14:30+00:00,E_1200353,Sent (mail and online)_complete,Application_1516032553
9,2017-01-02T12:14:30+00:00,E_1200353,Sent (mail and online)_complete,Offer_1104554024
10,2017-01-02T12:14:30+00:00,E_1200354,Complete application_complete,Application_1516032553
11,2017-01-02T12:14:30+00:00,E_1200357,Complete_complete,Application_1516032553


Offer_1318797754: 10 events, last: Complete_complete


,timestamp,event,activity,object
2,2016-12-30T02:09:11+00:00,E_1197680,Concept_complete,Application_1980650110
3,2017-01-02T12:14:45+00:00,E_1197683,Accepted_complete,Application_1980650110
4,2017-01-02T12:19:10+00:00,E_1197684,Create Offer_complete,Application_1980650110
5,2017-01-02T12:19:10+00:00,E_1197685,Created_complete,Application_1980650110
6,2017-01-02T12:19:10+00:00,E_1197685,Created_complete,Offer_1318797754
7,2017-01-02T12:19:20+00:00,E_1197686,Sent (mail and online)_complete,Application_1980650110
8,2017-01-02T12:19:20+00:00,E_1197686,Sent (mail and online)_complete,Offer_1318797754
9,2017-01-02T12:19:20+00:00,E_1197690,Complete_complete,Application_1980650110


Offer_1770894681: 34 events, last: Sent (online only)_complete


,timestamp,event,activity,object
26,2017-01-05T08:05:29+00:00,E_1161403,Validate application_complete,Application_1170192930
27,2017-01-05T08:05:29+00:00,E_1161406,Incomplete_complete,Application_1170192930
28,2017-02-01T10:07:53+00:00,E_1161423,Validating_complete,Application_1170192930
29,2017-02-01T10:07:56+00:00,E_1161424,Validate application_complete,Application_1170192930
30,2017-02-01T10:07:56+00:00,E_1161427,Incomplete_complete,Application_1170192930
31,2017-02-01T11:23:02+00:00,E_1161429,Create Offer_complete,Application_1170192930
32,2017-02-01T11:23:03+00:00,E_1161430,Created_complete,Application_1170192930
33,2017-02-01T11:23:42+00:00,E_1161431,Sent (online only)_complete,Application_1170192930


In [13]:
# Q3 clos(Offer, Sent): Sent (mail and online) is not an offer's last event.
q3 = bpi17.check("clos", "offer", activity="Sent (mail and online)_complete")
print(q3["obj"].nunique(), "offers")
q3.head()

1965 offers


,obj,event,activity,timestamp
0,Offer_1000145087,E_1044581,Sent (mail and online)_complete,2016-11-12T13:37:38+00:00
1,Offer_1000373613,E_180020,Sent (mail and online)_complete,2016-03-02T08:18:22+00:00
2,Offer_1003112260,E_464294,Sent (mail and online)_complete,2016-06-07T14:52:45+00:00
3,Offer_1003297346,E_97392,Sent (mail and online)_complete,2016-03-09T10:49:13+00:00
4,Offer_1005517344,E_1163449,Sent (mail and online)_complete,2016-12-19T19:20:13+00:00


In [14]:
# The same check for Returned.
returned = bpi17.check("clos", "offer", activity="Returned_complete")
print(returned["obj"].nunique(), "offers")
returned.head()

1172 offers


,obj,event,activity,timestamp
0,Offer_1000145087,E_1044593,Returned_complete,2016-11-24T10:28:54+00:00
1,Offer_1000373613,E_180032,Returned_complete,2016-03-07T08:58:08+00:00
2,Offer_1003112260,E_464306,Returned_complete,2016-06-13T07:06:54+00:00
3,Offer_1003297346,E_97400,Returned_complete,2016-03-14T13:11:13+00:00
4,Offer_100416010,E_516737,Returned_complete,2016-08-18T13:42:15+00:00


In [15]:
# Q5, Q5', Q6: carry-over from Sent (both channels) to Returned.
SENT = ["Sent (mail and online)_complete", "Sent (online only)_complete"]
for schema in ["carr_next", "carr", "carr_fwd"]:
    result = bpi17.check(schema, "offer", sources=SENT, targets="Returned_complete")
    print(f"{schema:<10} {len(result):>4} counterexamples   {bpi17.axiom(schema, otype='offer', sources=SENT, targets='Returned_complete')}")

carr_next     0 counterexamples   R_{Returned_complete}(offer) ⊑ ∃nextRole⁻.(R_{Sent (mail and online)_complete}(offer) ⊔ R_{Sent (online only)_complete}(offer))


carr          0 counterexamples   R_{Returned_complete}(offer) ⊑ ∃laterRole⁻.(R_{Sent (mail and online)_complete}(offer) ⊔ R_{Sent (online only)_complete}(offer))


carr_fwd    902 counterexamples   R_{Sent (mail and online)_complete}(offer) ⊔ R_{Sent (online only)_complete}(offer) ⊑ ∃laterRole.R_{Returned_complete}(offer)


## 3. Recording top-down checks and exporting to OWL

Each log records axioms used in top-down analysis for reproducibility and transparency. There is also the feature to export the ORO as an OWL ontology: `export_ttl()` exports the ORO of a log as .ttl file, including the core T-box for its object types, activities and role concepts, the bottom-up axioms with their tightest multiplicity bounds.

In [16]:
bpi17.recorded_axioms("top-down")

,schema,dl_axiom,mode,holds,counterexamples
0,"alt(offer, {Accepted_complete, Cancelled_compl...",O_offer ⊑ (=1 playsRole.(R_{Accepted_complete}...,top-down,False,4
1,"clos(offer, Sent (mail and online)_complete)",R_{Sent (mail and online)_complete}(offer) ⊑ R...,top-down,False,1965
2,"clos(offer, Returned_complete)",R_{Returned_complete}(offer) ⊑ R_end,top-down,False,1172
3,"carr_next(offer, {Sent (mail and online)_compl...",R_{Returned_complete}(offer) ⊑ ∃nextRole⁻.(R_{...,top-down,True,0
4,"carr(offer, {Sent (mail and online)_complete, ...",R_{Returned_complete}(offer) ⊑ ∃laterRole⁻.(R_...,top-down,True,0
5,"carr_fwd(offer, {Sent (mail and online)_comple...",R_{Sent (mail and online)_complete}(offer) ⊔ R...,top-down,False,902


In [17]:
ttl = bpi17.export_ttl()  # output/bpi17/oro.ttl
print(ttl.read_text()[:1500])

ORO written to output/bpi17/oro.ttl (196 schema instantiations)
@prefix ex: <http://www.example.com/> .
@prefix oro: <http://www.example.com/oro/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

oro:counterexamples a owl:AnnotationProperty .

oro:dl a owl:AnnotationProperty .

oro:holds a owl:AnnotationProperty .

<http://www.example.com/oro/log/bpi17_ocel> a owl:Ontology ;
    rdfs:label "Object Role Ontology of bpi17_ocel" ;
    oro:violated [ rdfs:label "carr_fwd(offer, {Sent (mail and online)_complete, Sent (online only)_complete}, Returned_complete)" ;
            oro:counterexamples "902"^^xsd:nonNegativeInteger ;
            oro:dl "R_{Sent (mail and online)_complete}(offer) ⊔ R_{Sent (online only)_complete}(offer) ⊑ ∃laterRole.R_{Returned_complete}(offer)" ;
            oro:holds false ;
            oro:mode "

## 4. Ontop: rewriting SPARL to SQL example

In [18]:
print(bpi17.translate_to_sql('''
PREFIX ocel: <https://stl.mie.utoronto.ca/ontologies/ocel/>
SELECT ?e ?a WHERE { ?e a ocel:Event ; ocel:hasActivity ?a . }
'''))

ans1(e, a)
CONSTRUCT [e, a] [a/RDF(https://stl.mie.utoronto.ca/ontologies/ocel/{}(activity1m2),IRI), e/RDF(http://www.example.com/{}(eid1m1),IRI)]
   NATIVE [activity1m2, eid1m1]
SELECT "v1"."activity" AS "activity1m2", "v1"."eid" AS "eid1m1"
FROM "log_events" "v1"
WHERE "v1"."activity" IS NOT NULL




## 5. Performance benchmark

Runtime and peak memory per stage for the four logs (4 fresh-process runs each; Ontop is
restarted per run). The published results and the comparison with OC-DECLARE, TOTeM and OCPQ are
in `experiments/`. This takes several minutes, so it is off by default.

In [19]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    from ocel_inference.benchmark import paper_datasets, run_benchmark

    benchmark = run_benchmark(paper_datasets(), runs=4, results_dir="experiments/results/oro_benchmark_rerun/")
    display(benchmark.summary().T, benchmark.scenario_summary())